In [ ]:
import sys
import json
import random
import hashlib
import shutil
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen

import torch
import transformers
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import files

ROOT = Path("/content")
SEED = 20261113
LABELS = ("yes", "no", "maybe")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED)

environment = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
}
print(json.dumps(environment, indent=2))
if DEVICE == "cpu":
    print("CPU is available for the smoke test. The full run will take longer.")

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/pubmedqa/pubmedqa/"
    "master/data/ori_pqal.json"
)
data_path = ROOT / "ori_pqal.json"

if not data_path.is_file():
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            downloaded = response.read()
        if not isinstance(json.loads(downloaded), dict):
            raise ValueError("Expected a JSON object keyed by PubMed ID.")
        data_path.write_bytes(downloaded)
    except Exception as exc:
        raise RuntimeError(
            "Download failed. Upload ori_pqal.json to /content using "
            "Colab's Files panel, then rerun this cell."
        ) from exc

data_bytes = data_path.read_bytes()
raw_data = json.loads(data_bytes)
assert isinstance(raw_data, dict) and len(raw_data) == 1000, (
    "Expected the official PQA-L file with 1,000 records."
)

records = []
for pmid, item in sorted(raw_data.items()):
    question = item["QUESTION"].strip()
    parts = item["CONTEXTS"]
    gold = item["final_decision"].strip().lower()
    assert question and isinstance(parts, list) and parts and gold in LABELS, pmid
    context = "\n".join(part.strip() for part in parts if part.strip())
    assert context, pmid
    records.append({
        "pmid": pmid, "question": question, "context": context, "gold": gold,
    })

# Stratify the 80/20 split by label. Gold labels never enter the prompts.
rng = random.Random(SEED)
dev, test = [], []
for label in LABELS:
    group = [r for r in records if r["gold"] == label]
    rng.shuffle(group)
    n_test = round(len(group) * 0.20)
    test.extend(group[:n_test])
    dev.extend(group[n_test:])

rng.shuffle(dev)
rng.shuffle(test)

assert not ({r["pmid"] for r in dev} & {r["pmid"] for r in test})
majority = max(LABELS, key=lambda label: sum(r["gold"] == label for r in dev))

run_dir = ROOT / ("pubmedqa_run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
run_dir.mkdir()
(run_dir / "split_ids.json").write_text(json.dumps({
    "seed": SEED,
    "development_ids": [r["pmid"] for r in dev],
    "test_ids": [r["pmid"] for r in test],
}, indent=2))

dataset_report = {
    "source": DATA_URL,
    "sha256": hashlib.sha256(data_bytes).hexdigest(),
    "total": len(records),
    "development": len(dev),
    "test": len(test),
    "label_counts": dict(Counter(r["gold"] for r in records)),
    "development_majority": majority,
}
(run_dir / "dataset_validation.json").write_text(json.dumps(dataset_report, indent=2))
print(json.dumps(dataset_report, indent=2))
print("Run directory:", run_dir)

In [ ]:
import re

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()

CONDITIONS = ("direct", "context", "context_abstain")

def make_prompt(question, context, condition):
    instruction = "Answer the biomedical research question with yes, no, or maybe."
    if condition == "context_abstain":
        instruction += " Use abstain if the abstract provides insufficient evidence."
    instruction += " Output only the answer."
    if condition != "direct":
        instruction += f"\n\nAbstract:\n{context}"
    instruction += f"\n\nQuestion: {question}"
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,
    )

def parse_answer(text, condition):
    match = re.fullmatch(r"(yes|no|maybe|abstain)[.!]?", text.strip().lower())
    if not match:
        return "invalid"
    answer = match.group(1)
    return "invalid" if answer == "abstain" and condition != "context_abstain" else answer

@torch.inference_mode()
def infer(batch, condition):
    prompts = [make_prompt(r["question"], r["context"], condition) for r in batch]
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, add_special_tokens=False
    ).to(DEVICE)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    tokens = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    new_tokens = tokens[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True), elapsed

assert parse_answer("Yes.", "direct") == "yes"
assert parse_answer("abstain", "direct") == "invalid"
assert parse_answer("abstain", "context_abstain") == "abstain"

for condition in CONDITIONS:
    replies, seconds = infer(dev[:2], condition)
    print(condition, [(r, parse_answer(r, condition)) for r in replies],
          f"({seconds:.2f}s per batch)")

environment.update({
    "model": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy",
    "parser": "exact label with optional final period or exclamation mark",
})
(run_dir / "run_metadata.json").write_text(json.dumps(environment, indent=2))

In [ ]:
rows = [
    {
        "pmid": r["pmid"], "condition": "majority", "gold": r["gold"],
        "prediction": majority, "raw_response": "", "batch_seconds_per_item": None,
    }
    for r in test
]

for condition in CONDITIONS:
    for start in range(0, len(test), BATCH_SIZE):
        batch = test[start:start + BATCH_SIZE]
        replies, seconds = infer(batch, condition)
        assert len(replies) == len(batch)
        for record, reply in zip(batch, replies):
            rows.append({
                "pmid": record["pmid"],
                "condition": condition,
                "gold": record["gold"],
                "prediction": parse_answer(reply, condition),
                "raw_response": reply,
                "batch_seconds_per_item": seconds / len(batch),
            })
        done = start + len(batch)
        if done % (5 * BATCH_SIZE) == 0 or done == len(test):
            print(f"{condition}: {done}/{len(test)}", flush=True)

predictions = pd.DataFrame(rows)
assert len(predictions) == 4 * len(test)
assert not predictions.duplicated(["condition", "pmid"]).any()

expected = {r["pmid"]: r["gold"] for r in test}
for _, group in predictions.groupby("condition"):
    assert dict(zip(group["pmid"], group["gold"])) == expected

predictions.to_csv(run_dir / "predictions.csv", index=False)
print(f"Saved {len(predictions)} prediction rows in {run_dir}")

In [ ]:
predictions = pd.read_csv(run_dir / "predictions.csv", dtype={"pmid": str})
summaries, class_rows, error_rows = [], [], []

for condition, group in predictions.groupby("condition", sort=False):
    correct = group["prediction"].eq(group["gold"])
    answered = group["prediction"].isin(LABELS)
    report = classification_report(
        group["gold"], group["prediction"], labels=list(LABELS),
        output_dict=True, zero_division=0,
    )
    summaries.append({
        "condition": condition,
        "n": len(group),
        "accuracy": correct.mean(),
        "macro_f1": report["macro avg"]["f1-score"],
        "coverage": answered.mean(),
        "selective_accuracy": correct[answered].mean() if answered.any() else None,
        "abstention_rate": group["prediction"].eq("abstain").mean(),
        "invalid_rate": group["prediction"].eq("invalid").mean(),
        "mean_batch_seconds_per_item": group["batch_seconds_per_item"].mean(),
    })
    for label in LABELS:
        class_rows.append({"condition": condition, "label": label, **report[label]})
    error_rows.append(group.loc[~correct])

metrics = pd.DataFrame(summaries)
metrics.to_csv(run_dir / "metrics_by_condition.csv", index=False)
pd.DataFrame(class_rows).to_csv(run_dir / "per_class_metrics.csv", index=False)
pd.concat(error_rows).to_csv(run_dir / "error_analysis.csv", index=False)
display(metrics.round(4))

paired = predictions.pivot(index="pmid", columns="condition", values="prediction")
gold = predictions.drop_duplicates("pmid").set_index("pmid")["gold"].reindex(paired.index)
direct_ok = paired["direct"].eq(gold)
context_ok = paired["context"].eq(gold)
print("Context correct / direct wrong:", int((context_ok & ~direct_ok).sum()))
print("Direct correct / context wrong:", int((direct_ok & ~context_ok).sum()))
paired.assign(gold=gold).to_csv(run_dir / "paired_predictions.csv")

ax = metrics.set_index("condition")[["accuracy", "macro_f1", "coverage"]].plot.bar(
    figsize=(9, 4), ylim=(0, 1), rot=15
)
ax.set_ylabel("Score")
ax.figure.tight_layout()
ax.figure.savefig(run_dir / "condition_comparison.png", dpi=180)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (condition, group) in zip(axes.flat, predictions.groupby("condition", sort=False)):
    ConfusionMatrixDisplay.from_predictions(
        group["gold"], group["prediction"],
        labels=[*LABELS, "abstain", "invalid"], ax=ax, colorbar=False,
    )
    ax.set_title(condition)
fig.tight_layout()
fig.savefig(run_dir / "confusion_matrices.png", dpi=180)
plt.show()

archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
files.download(archive)